In [4]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import r2_score


# 1. Cargar

df = pd.read_csv("kc_house_data.csv")

print("Dimensiones:", df.shape)
df.head()

Dimensiones: (21613, 21)


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [6]:
# Variable dependiente a predecir
y = df['price']

# Variables independientes
X = df[[
    'bedrooms',
    'bathrooms',
    'sqft_living',
    'sqft_lot',
    'floors',
    'waterfront',
    'view',
    'condition',
    'grade',
    'sqft_basement',
    'yr_built',
    'yr_renovated',
    'lat',
    'long',
    'sqft_living15',
    'sqft_lot15'
]]

In [8]:
# 2. Agregar columna de unos

X_mat = np.column_stack((np.ones(len(X)), X))

# Convertir a matrices numpy
X_mat = np.array(X_mat)
y_mat = np.array(y)

# 3. Beta = (X'X)^(-1)X'Y


beta = np.linalg.inv(X_mat.T @ X_mat) @ X_mat.T @ y_mat

coeficientes = pd.DataFrame({
    'Variable':['Intercepto'] + list(X.columns),
    'Coeficiente':beta
})

coeficientes

,Variable,Coeficiente
0,Intercepto,-3.686262e+07
1,bedrooms,-3.415167e+04
2,bathrooms,4.216171e+04
3,sqft_living,1.795138e+02
4,sqft_lot,1.274163e-01
5,floors,7.606881e+02
6,waterfront,5.878472e+05
7,view,4.942923e+04
8,condition,3.103180e+04
9,grade,9.721937e+04


In [10]:
# 4. Predicciones

y_pred = X_mat @ beta


# 5. R²

r2 = r2_score(y, y_pred)

n = len(y)
k = X.shape[1]

r2_adj = 1 - ((1-r2)*(n-1)/(n-k-1))

print("R² =", round(r2,4))
print("R² Ajustado =", round(r2_adj,4))

R² = 0.6954
R² Ajustado = 0.6952


In [12]:
# 6. Reporte estadístico


X_sm = sm.add_constant(X)

modelo = sm.OLS(y, X_sm).fit()

print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.695
Model:                            OLS   Adj. R-squared:                  0.695
Method:                 Least Squares   F-statistic:                     3082.
Date:                Tue, 30 Jun 2026   Prob (F-statistic):               0.00
Time:                        08:29:37   Log-Likelihood:            -2.9476e+05
No. Observations:               21613   AIC:                         5.895e+05
Df Residuals:                   21596   BIC:                         5.897e+05
Df Model:                          16                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const         -3.686e+07    1.6e+06    -23.105

In [14]:
# 7. Valores p


pvalues = pd.DataFrame({
    'Variable': modelo.pvalues.index,
    'p_value': modelo.pvalues.values
})

pvalues.sort_values('p_value')

,Variable,p_value
13,lat,0.000000e+00
3,sqft_living,0.000000e+00
9,grade,0.000000e+00
11,yr_built,1.133690e-244
6,waterfront,1.197866e-241
0,const,1.060633e-116
7,view,6.019411e-116
1,bedrooms,1.750369e-71
8,condition,1.508236e-39
2,bathrooms,9.386464e-38


In [18]:
# Variables con p > 0.05

variables_significativas = modelo.pvalues[
    modelo.pvalues < 0.05
].index.tolist()

variables_significativas.remove('const')

print("Variables significativas:")
print(variables_significativas)

Variables significativas:
['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'waterfront', 'view', 'condition', 'grade', 'sqft_basement', 'yr_built', 'yr_renovated', 'lat', 'long', 'sqft_living15', 'sqft_lot15']


In [20]:
# 8. Modelo final


X_final = df[variables_significativas]

X_final_sm = sm.add_constant(X_final)

modelo_final = sm.OLS(y, X_final_sm).fit()

print(modelo_final.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.695
Model:                            OLS   Adj. R-squared:                  0.695
Method:                 Least Squares   F-statistic:                     3287.
Date:                Tue, 30 Jun 2026   Prob (F-statistic):               0.00
Time:                        08:29:59   Log-Likelihood:            -2.9476e+05
No. Observations:               21613   AIC:                         5.895e+05
Df Residuals:                   21597   BIC:                         5.897e+05
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const         -3.692e+07   1.57e+06    -23.516

In [22]:
# 9. Ecuación final


coef = modelo_final.params

print("\nECUACIÓN FINAL:\n")

ecuacion = f"Precio = {coef.iloc[0]:.2f}"

for variable, valor in coef.iloc[1:].items():
    ecuacion += f" + ({valor:.4f})*{variable}"

print(ecuacion)


ECUACIÓN FINAL:

Precio = -36922319.02 + (-34164.4966)*bedrooms + (42341.0922)*bathrooms + (179.6412)*sqft_living + (0.1272)*sqft_lot + (587854.2714)*waterfront + (49444.0313)*view + (30998.1762)*condition + (97258.3187)*grade + (-33.2686)*sqft_basement + (-2453.0853)*yr_built + (21.5718)*yr_renovated + (561297.2535)*lat + (-117371.0868)*long + (27.3428)*sqft_living15 + (-0.3938)*sqft_lot15
